# Defining your own dataset

This notebook demonstrates how to optimize over your own labelled dataset. It will include metadata fields on your documents which can be included in your custom defined search queries.

# Dataset

The dataset to reference is `../resources/custom/custom_corpus.json`, with corresponding `../resources/custom/custom_queries.json` and `../resources/custom/custom_qrels.json`. Let's load and inspect them.

In [1]:
import json

corpus_file = "../resources/custom/custom_corpus.json"
queries_file = "../resources/custom/custom_queries.json"
qrels_file = "../resources/custom/custom_qrels.json"

with open(corpus_file) as file:
    corpus = json.load(file)

with open(queries_file) as file:
    queries = json.load(file)

with open(qrels_file) as file:
    qrels = json.load(file)

In [2]:
from redis_retrieval_optimizer.bayes_study import run_bayes_study
from typing import Any
from redisvl.utils.vectorize import BaseVectorizer

config_path = "./custom_data_config.yaml"
redis_url = "redis://localhost:6379"

def custom_corpus_processor(corpus: dict[dict[str:Any]], emb_model: BaseVectorizer) -> list[dict[str:str]]:
    """ Custom processing logic for the corpus
    Args:
        corpus (dict): The corpus data. This is read in from a JSON file containing a single JSON
            object.
        emb_model: The RedisVL embedding model to use for vectorization. These are specified in the
            study_config.yaml file and may be HuggingFace model names, or one of the supported 
            embeddings APIs. see <> for a list.
    Returns:
        list: A list of documents to be indexed in Redis. Each document should have the following
            fields:
            - _id: The unique identifier for the document.

            also have the fields listed under 'additional_fields' defined in the study_config.yaml
            file. In this example they are:
            - product_name: The name of the product.
            - price: The price of the product.
            - in_stock: A boolean indicating if the product is in stock, cast to string, read as a Redis TAG
            - vector: The vector representation of the product name.
    """
    corpus_data = []

    texts = []
    for key in corpus:
        texts.append(corpus[key]['product_name'])
    embeddings = emb_model.embed_many(texts, as_buffer=True)

    for key, embedding in zip(corpus, embeddings):
        doc = {
            "_id": key,
            "product_name": corpus[key]["product_name"],
            "price": corpus[key]["price"],
            "in_stock": "TRUE" if corpus[key]["in_stock"] else "FALSE",
            "vector_embedding": embedding
        }
        corpus_data.append(doc)

    return corpus_data


/Users/justin.cechmanek/.pyenv/versions/ret-opt/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# because we are defining a custom search method that will utilize RedisVL FilterExpressions
# we need to define a custom search method map that tells the retrieval optimizer how to
# handle the custom search method.

from redis_retrieval_optimizer.schema import SearchMethodInput, SearchMethodOutput

from redis_retrieval_optimizer.search_methods import (
    gather_vector_results,
    gather_hybrid_results,
    gather_bm25_results,
)


def custom_gather_filter_query_results( search_method_input: SearchMethodInput) -> SearchMethodOutput:
    """ Custom search method that uses RedisVL FilterExpressions to filter results based on the
    query.
    Args:
        search_method_input (SearchMethodInput): The input to the search method, which includes
        the query, corpus, and Redis URL.
    Returns:
        SearchMethodOutput
    """

    return gather_vector_results(search_method_input)
    # This function will use RedisVL FilterExpressions to filter results based on the query.



custom_search_method_map = {
    "vector": gather_vector_results, # this is predefined, so we can just import and ust it
    "hybrid": gather_hybrid_results,
    "bm25": gather_bm25_results,
    "filter_query": custom_gather_filter_query_results # this is our custom search method
    }


In [4]:
# run the study
metrics = run_bayes_study(
    config_path=config_path,
    redis_url=redis_url,
    #corpus_processor=eval_beir.process_corpus,
    corpus_processor=custom_corpus_processor,
    search_method_map=custom_search_method_map,
)

[I 2025-05-30 16:55:14,212] A new study created in memory with name: test


16:55:14 redisvl.index.index INFO   Index already exists, overwriting.
16:55:17 datasets INFO   PyTorch version 2.7.0 available.
16:55:17 sentence_transformers.SentenceTransformer INFO   Use pytorch device_name: mps
16:55:17 sentence_transformers.SentenceTransformer INFO   Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2


Batches: 100%|██████████| 1/1 [00:00<00:00,  6.53it/s]

16:55:20 root INFO   Data indexed total_indexing_time=0.0s, num_docs=20


16:55:22 root INFO   Saving metrics for study: 94fc598e-057c-4de0-90c3-3bdc326fc1af, METRICS={'search_method': ['bm25'], 'total_indexing_time': [0.0], 'avg_query_time': [0.000664675235748291], 'model': ['sentence-transformers/all-MiniLM-L6-v2'], 'model_dim': [384], 'ret_k': [8], 'recall@k': [0.4721428571428571], 'ndcg@k': [0.5075095878967448], 'f1@k': [0.49603174603174593], 'precision': [0.5958333333333333], 'algorithm': ['hnsw'], 'ef_construction': [100], 'ef_runtime': [30], 'm': [16], 'distance_metric': ['cosine'], 'vector_data_type': ['float32']}


...

16:55:33 redisvl.index.index INFO   Index already exists, overwriting.
16:55:33 sentence_transformers.SentenceTransformer INFO   Use pytorch device_name: mps
16:55:33 sentence_transformers.SentenceTransformer INFO   Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2


Batches: 100%|██████████| 1/1 [00:00<00:00, 21.17it/s]

16:55:34 root INFO   Data indexed total_indexing_time=0.001s, num_docs=20
16:55:34 root INFO   Saving metrics for study: 94fc598e-057c-4de0-90c3-3bdc326fc1af, METRICS={'search_method': ['bm25', 'bm25', 'bm25', 'bm25', 'bm25', 'bm25', 'bm25', 'bm25', 'bm25', 'bm25'], 'total_indexing_time': [0.0, 0.0, 0.0, 0.001, 0.0, 0.001, 0.0, 0.001, 0.001, 0.001], 'avg_query_time': [0.000664675235748291, 0.0009545207023620606, 0.0009898781776428224, 0.0009114384651184082, 0.0013636469841003418, 0.001991379261016846, 0.0009043574333190918, 0.0006473779678344727, 0.0005701541900634766, 0.0006150603294372559], 'model': ['sentence-transformers/all-MiniLM-L6-v2', 'sentence-transformers/all-MiniLM-L6-v2', 'sentence-transformers/all-MiniLM-L6-v2', 'sentence-transformers/all-MiniLM-L6-v2', 'sentence-transformers/all-MiniLM-L6-v2', 'sentence-transformers/all-MiniLM-L6-v2', 'sentence-transformers/all-MiniLM-L6-v2', 'sentence-transformers/all-MiniLM-L6-v2', 'sentence-transformers/all-MiniLM-L6-v2', 'sentence-tr


[I 2025-05-30 16:55:34,727] Trial 9 finished with value: 0.001 and parameters: {'model_info': {'type': 'hf', 'model': 'sentence-transformers/all-MiniLM-L6-v2', 'dim': 384, 'embedding_cache_name': 'vec-cache', 'dtype': 'float32'}, 'search_method': 'bm25', 'algorithm': 'hnsw', 'var_dtype': 'float32', 'distance_metric': 'cosine', 'ret_k': 6, 'ef_runtime': 30, 'ef_construction': 100, 'm': 8}. Best is trial 3 with value: 0.001.


16:55:34 root INFO   Completed Bayesian optimization... 


16:55:34 root INFO   Best Configuration: 3: {'model_info': {'type': 'hf', 'model': 'sentence-transformers/all-MiniLM-L6-v2', 'dim': 384, 'embedding_cache_name': 'vec-cache', 'dtype': 'float32'}, 'search_method': 'bm25', 'algorithm': 'hnsw', 'var_dtype': 'float32', 'distance_metric': 'cosine', 'ret_k': 5, 'ef_runtime': 50, 'ef_construction': 100, 'm': 8}:


16:55:34 root INFO   Best Score: [0.001]




In [5]:
metrics.sort_values(by='search_method')

,search_method,total_indexing_time,avg_query_time,model,model_dim,ret_k,recall@k,ndcg@k,f1@k,precision,algorithm,ef_construction,ef_runtime,m,distance_metric,vector_data_type
0,bm25,0.000,0.000665,sentence-transformers/all-MiniLM-L6-v2,384,8,0.472143,0.50751,0.496032,0.595833,hnsw,100,30,16,cosine,float32
1,bm25,0.000,0.000955,sentence-transformers/all-MiniLM-L6-v2,384,6,0.472143,0.50751,0.496032,0.595833,hnsw,300,10,16,cosine,float32
2,bm25,0.000,0.000990,sentence-transformers/all-MiniLM-L6-v2,384,6,0.472143,0.50751,0.496032,0.595833,hnsw,100,10,16,cosine,float32
3,bm25,0.001,0.000911,sentence-transformers/all-MiniLM-L6-v2,384,5,0.472143,0.50751,0.496032,0.595833,hnsw,100,50,8,cosine,float32
4,bm25,0.000,0.001364,sentence-transformers/all-MiniLM-L6-v2,384,7,0.472143,0.50751,0.496032,0.595833,hnsw,100,30,16,cosine,float32
5,bm25,0.001,0.001991,sentence-transformers/all-MiniLM-L6-v2,384,5,0.472143,0.50751,0.496032,0.595833,hnsw,100,50,16,cosine,float32
6,bm25,0.000,0.000904,sentence-transformers/all-MiniLM-L6-v2,384,8,0.472143,0.50751,0.496032,0.595833,hnsw,300,50,64,cosine,float32
7,bm25,0.001,0.000647,sentence-transformers/all-MiniLM-L6-v2,384,10,0.472143,0.50751,0.496032,0.595833,hnsw,100,50,16,cosine,float32
8,bm25,0.001,0.000570,sentence-transformers/all-MiniLM-L6-v2,384,8,0.472143,0.50751,0.496032,0.595833,hnsw,300,30,8,cosine,float32
9,bm25,0.001,0.000615,sentence-transformers/all-MiniLM-L6-v2,384,6,0.472143,0.50751,0.496032,0.595833,hnsw,100,30,8,cosine,float32
